# UACB-604 · Lecture 2.2 — Inside a real Illumina FASTQ file

We stream the first 20,000 read pairs of a real Illumina HiSeq 2000 run, **SRR1039508**, without downloading the whole 1.2 GB file.
It is from the human airway RNA-seq study PRJNA229998, the same samples used in many RNA-seq tutorials.

**You will:**
1. Read FASTQ records and decode Phred quality characters
2. Plot quality along the read and explain the drop with **phasing**
3. Find the `N` bases and their quality
4. Compute **expected errors** per read and filter reads
5. Check that read 1 and read 2 form pairs

In [ ]:
import gzip, urllib.request, itertools, collections
import numpy as np, matplotlib.pyplot as plt

BASE = "https://ftp.sra.ebi.ac.uk/vol1/fastq/SRR103/008/SRR1039508/"

def stream_fastq(url, n_reads):
    # Yield (header, sequence, quality) for the first n_reads of a gzipped FASTQ on the web
    req = urllib.request.Request(url, headers={"User-Agent": "uacb604-course"})
    with gzip.GzipFile(fileobj=urllib.request.urlopen(req, timeout=60)) as fh:
        lines = (l.decode().rstrip("\n") for l in fh)
        for _ in range(n_reads):
            h, s, plus, q = (next(lines) for _ in range(4))
            yield h, s, q

r1 = list(stream_fastq(BASE + "SRR1039508_1.fastq.gz", 20_000))
r2 = list(stream_fastq(BASE + "SRR1039508_2.fastq.gz", 20_000))
for h, s, q in r1[:2]:
    print(h); print(s); print("+"); print(q); print()

## 1. Decoding quality characters
Each quality character is one byte. Its ASCII code minus **33** is the Phred score $Q$ ("Phred+33" encoding), and $p = 10^{-Q/10}$.

In [ ]:
h, s, q = r1[0]
print(" pos base char ASCII  Q   p(error)")
for i in range(12, 26):
    Q = ord(q[i]) - 33
    print(f"{i+1:4d}   {s[i]}    {q[i]}    {ord(q[i]):3d}  {Q:2d}  {10**(-Q/10):.4f}")

The `N` bases (the machine could not decide) all carry `#` = ASCII 35 → Q2 → error probability 0.63. The base caller is honest about not knowing.

## 2. Quality along the read

In [ ]:
Q = np.array([[ord(c) - 33 for c in q] for _, _, q in r1])      # reads × positions (all reads are 63 bases)
print("matrix shape:", Q.shape, " highest score:", Q.max(), " lowest:", Q.min())

pos = np.arange(1, Q.shape[1] + 1)
plt.figure(figsize=(10, 4))
plt.fill_between(pos, np.percentile(Q, 10, axis=0), np.percentile(Q, 90, axis=0), alpha=0.3, label="10th–90th percentile")
plt.plot(pos, np.median(Q, axis=0), lw=2, label="median")
plt.axhline(30, color="red", ls="--", lw=1, label="Q30")
plt.xlabel("position in read (cycle)"); plt.ylabel("Phred quality"); plt.ylim(0, 42); plt.legend(); plt.show()

Quality falls towards the end of the read. One main cause is **phasing**: in each cycle a small fraction $\varepsilon$ of the molecules in a cluster fail to extend (or extend twice) and fall out of step. After $n$ cycles the fraction still in step is
$$(1-\varepsilon)^n$$
so the signal gets weaker and the out-of-step molecules add noise.

In [ ]:
n = np.arange(0, 301)
for eps in (0.001, 0.003, 0.005):
    plt.plot(n, (1 - eps) ** n, label=f"ε = {eps}")
plt.xlabel("cycle"); plt.ylabel("fraction of molecules in step"); plt.legend(); plt.show()

✏️ **Exercise 1.** With ε = 0.005, after how many cycles is only half of the cluster in step? Solve $(1-\varepsilon)^n = 0.5$ with logarithms, then check on the plot.

In [ ]:
# your code here

## 3. Base composition and N

In [ ]:
seqs = [s for _, s, _ in r1]
comp = {b: [sum(sq[i] == b for sq in seqs) / len(seqs) for i in range(len(seqs[0]))] for b in "ACGTN"}
for b, c in comp.items():
    plt.plot(pos, c, label=b)
plt.xlabel("position"); plt.ylabel("fraction of reads"); plt.legend(); plt.show()
print("fraction of all bases that are N:", round(sum(sq.count("N") for sq in seqs) / sum(map(len, seqs)), 5))

✏️ **Exercise 2.** The first ~10 positions show uneven A/C/G/T. RNA-seq libraries are made with *random hexamer* primers, which are not perfectly random in practice. Why would that show up only at the start of reads?

## 4. Expected errors per read
The expected number of errors in a read is the sum of its error probabilities (Lecture 1.1, expected value):
$$EE = \sum_{i} 10^{-Q_i/10}$$
A common filter keeps reads with $EE \le 1$.

In [ ]:
EE = (10 ** (-Q / 10)).sum(axis=1)
plt.hist(np.clip(EE, 0, 5), bins=50); plt.axvline(1, color="red", ls="--")
plt.xlabel("expected errors per read (clipped at 5)"); plt.ylabel("reads"); plt.show()
print(f"median EE = {np.median(EE):.3f};  reads with EE ≤ 1: {np.mean(EE <= 1):.1%}")

## 5. Pairs
In paired-end sequencing each DNA fragment is read from both ends. Read 1 and read 2 come in the same order in two files, with matching names.

In [ ]:
same = sum(a[0].split()[0] == b[0].split()[0] for a, b in zip(r1, r2))
print(r1[0][0]); print(r2[0][0]); print(f"{same:,} of {len(r1):,} pairs have matching names")

✏️ **Exercise 3.** Suppose a fragment is 250 bp long, and each read is 63 bp. Draw a fragment with its two reads. Do the reads overlap? What would change with 2 × 150 bp reads?

---
**Further reading:** Pevsner, ch. 9 (analysis of next-generation sequence data) · Buffalo, ch. 10 (working with sequence data).